# LeWM Duckietown Retrain (Exploratory Data)\n\nThis notebook is the current **active** path for exploratory-data experiments.\n\nWorkflow:\n1. Pull `duckie_explore.h5`\n2. Run the encoder-based `obs -> action` probe gate\n3. Proceed to training only if steering `val R² < 0.6`\n

In [ ]:
# Config\nDATA_NEW_S3 = 's3://leworldduckie/data/duckie_explore.h5'\nDATA_OLD_S3 = 's3://leworldduckie/data/duckietown_100k.h5'\nDATA_NEW_LOCAL = '/content/duckie_explore.h5'\nDATA_OLD_LOCAL = '/content/duckietown_100k.h5'\n\n# Use the fs6 checkpoint (or update if newer)\nCKPT_S3 = 's3://leworldduckie/training/runs/notebook/checkpoint_best.pt'\nCKPT_LOCAL = '/content/lewm_checkpoint.pt'\n\n# Probe gate threshold\nSTEER_R2_GATE = 0.6\n

In [ ]:
# Colab setup\nimport os, subprocess, sys\n\ndef sh(cmd):\n    print('>', cmd)\n    subprocess.check_call(['bash', '-lc', cmd])\n\nsh('pip install -q boto3 h5py scikit-learn torch torchvision')\nsh('git clone --depth 1 https://github.com/giuliovv/leworldduckie.git /content/leworldduckie || true')\n

In [ ]:
# Download datasets + checkpoint from S3\nimport boto3\nfrom urllib.parse import urlparse\n\ndef s3_download(uri, local):\n    u = urlparse(uri)\n    boto3.client('s3', region_name='us-east-1').download_file(u.netloc, u.path.lstrip('/'), local)\n    print('downloaded', uri, '->', local)\n\ns3_download(DATA_NEW_S3, DATA_NEW_LOCAL)\ns3_download(DATA_OLD_S3, DATA_OLD_LOCAL)\ns3_download(CKPT_S3, CKPT_LOCAL)\n

In [ ]:
# Probe gate (encoder mode default).\n# IMPORTANT: Do not start retraining unless steering val R² < 0.6\ncmd = '''\ncd /content/leworldduckie && python src/probe_obs_to_action.py \\n  --mode encoder \\n  --ckpt {ckpt} \\n  --data {new_data} \\n  --baseline-data {old_data} \\n  --max-samples 50000 \\n  --epochs 8 \\n  --batch-size 256\n'''.format(ckpt=CKPT_LOCAL, new_data=DATA_NEW_LOCAL, old_data=DATA_OLD_LOCAL)\n\nimport subprocess\nsubprocess.check_call(['bash', '-lc', cmd])\n

## Training (run only if gate passes)\n\nIf steering `val R² < 0.6`, continue with your training notebook cells or scripts.\n\nExample script entrypoint:\n```bash\ncd /content/leworldduckie\npython src/train.py\n```\n